<a href="https://colab.research.google.com/github/Dipupo14/August-Cohort/blob/master/Copy_of_Capstone_Project_Grp_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Traffic Accident Severity Prediction
## TechCrush AI/ML Track Cohort 8-Group 5

## 1. Data Acquisition

The dataset used for this project is the US Accidents (March 2023) dataset created by Sobhan Moosavi, sourced from Kaggle. The full dataset has over 7 million rows and 46 columns covering road accidents across the United States between 2016 and 2023. Each row is a single accident record with details about location, time, weather conditions, road features, and how severely the accident affected traffic flow.

Severity runs from 1 to 4. A rating of 1 means the accident had little impact on traffic; 4 means it caused significant disruption. That severity rating is what our model learns to predict.

The full file exceeds what Google Colab can load into memory without crashing, so we sampled 500,000 rows randomly using random_state=42. Anyone running this notebook gets the same 500,000 rows every time.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Capstone Project/US_Accidents_Sample.csv')
print(df.shape)
print(df.head())

Mounted at /content/drive
(500000, 46)
    ID   Source  Severity           Start_Time             End_Time  \
0  A-1  Source2         3  2016-02-08 05:46:00  2016-02-08 11:00:00   
1  A-2  Source2         2  2016-02-08 06:07:59  2016-02-08 06:37:59   
2  A-3  Source2         2  2016-02-08 06:49:27  2016-02-08 07:19:27   
3  A-4  Source2         3  2016-02-08 07:23:34  2016-02-08 07:53:34   
4  A-5  Source2         2  2016-02-08 07:39:07  2016-02-08 08:09:07   

   Start_Lat  Start_Lng  End_Lat  End_Lng  Distance(mi)  ... Roundabout  \
0  39.865147 -84.058723      NaN      NaN          0.01  ...      False   
1  39.928059 -82.831184      NaN      NaN          0.01  ...      False   
2  39.063148 -84.032608      NaN      NaN          0.01  ...      False   
3  39.747753 -84.205582      NaN      NaN          0.01  ...      False   
4  39.627781 -84.188354      NaN      NaN          0.01  ...      False   

  Station   Stop Traffic_Calming Traffic_Signal Turning_Loop Sunrise_Sunset  \
0   

In [ ]:
import pandas as pd

# Load the accident dataset from Google Drive
df = pd.read_csv(
    '/content/drive/MyDrive/Capstone Project/US_Accidents_Sample.csv'
)

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (500000, 46)
    ID   Source  Severity           Start_Time             End_Time  \
0  A-1  Source2         3  2016-02-08 05:46:00  2016-02-08 11:00:00   
1  A-2  Source2         2  2016-02-08 06:07:59  2016-02-08 06:37:59   
2  A-3  Source2         2  2016-02-08 06:49:27  2016-02-08 07:19:27   
3  A-4  Source2         3  2016-02-08 07:23:34  2016-02-08 07:53:34   
4  A-5  Source2         2  2016-02-08 07:39:07  2016-02-08 08:09:07   

   Start_Lat  Start_Lng  End_Lat  End_Lng  Distance(mi)  ... Roundabout  \
0  39.865147 -84.058723      NaN      NaN          0.01  ...      False   
1  39.928059 -82.831184      NaN      NaN          0.01  ...      False   
2  39.063148 -84.032608      NaN      NaN          0.01  ...      False   
3  39.747753 -84.205582      NaN      NaN          0.01  ...      False   
4  39.627781 -84.188354      NaN      NaN          0.01  ...      False   

  Station   Stop Traffic_Calming Traffic_Signal Turning_Loop Sunrise_Sunset  \
0   False  Fals

In [ ]:
print(df['Severity'].value_counts())

Severity
2    312159
3    187173
1       388
4       280
Name: count, dtype: int64


The severity distribution shows a significant class imbalance. Severity 2 accounts for roughly 312,000 of the 500,000 records while severity 1 and 4 combined make up less than 700 records. This imbalance will affect how the models perform on minority classes and needs to be accounted for during modelling.

In [ ]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 46 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   ID                     500000 non-null  object 
 1   Source                 500000 non-null  object 
 2   Severity               500000 non-null  int64  
 3   Start_Time             500000 non-null  object 
 4   End_Time               500000 non-null  object 
 5   Start_Lat              500000 non-null  float64
 6   Start_Lng              500000 non-null  float64
 7   End_Lat                0 non-null       float64
 8   End_Lng                0 non-null       float64
 9   Distance(mi)           500000 non-null  float64
 10  Description            500000 non-null  object 
 11  Street                 500000 non-null  object 
 12  City                   499978 non-null  object 
 13  County                 500000 non-null  object 
 14  State                  500000 non-nu

The df.info() output shows 46 columns across 500,000 rows. A few things stand out immediately.

End_Lat and End_Lng have zero non-null values, meaning both columns are completely empty in our sample. Wind_Chill(F) has only 62,359 values and Precipitation(in) has just 51,946, both out of 500,000. These columns will need attention during cleaning. Start_Time and End_Time are stored as object type rather than datetime, which also needs fixing before any time-based features can be extracted.

| Column | Data Type | Description |
|---|---|---|
| ID | object | Unique identifier for each accident record |
| Source | object | Source of the accident report |
| Severity | int64 | Accident severity rating from 1 (minor) to 4 (severe) |
| Start_Time | object | Date and time the accident started |
| End_Time | object | Date and time the accident ended |
| Start_Lat | float64 | Latitude of the accident start point |
| Start_Lng | float64 | Longitude of the accident start point |
| End_Lat | float64 | Latitude of the accident end point (empty in this sample) |
| End_Lng | float64 | Longitude of the accident end point (empty in this sample) |
| Distance(mi) | float64 | Length of road affected by the accident in miles |
| Description | object | Natural language description of the accident |
| Street | object | Street name where the accident occurred |
| City | object | City where the accident occurred |
| County | object | County where the accident occurred |
| State | object | US state where the accident occurred |
| Zipcode | object | Zip code of the accident location |
| Country | object | Country (all US in this dataset) |
| Timezone | object | Timezone of the accident location |
| Airport_Code | object | Nearest airport code to the accident |
| Weather_Timestamp | object | Time the weather reading was recorded |
| Temperature(F) | float64 | Air temperature in Fahrenheit at time of accident |
| Wind_Chill(F) | float64 | Wind chill temperature in Fahrenheit |
| Humidity(%) | float64 | Humidity percentage at time of accident |
| Pressure(in) | float64 | Atmospheric pressure in inches |
| Visibility(mi) | float64 | Visibility in miles at time of accident |
| Wind_Direction | object | Direction of wind at time of accident |
| Wind_Speed(mph) | float64 | Wind speed in miles per hour |
| Precipitation(in) | float64 | Precipitation amount in inches |
| Weather_Condition | object | General weather condition description |
| Amenity | bool | Whether an amenity was nearby |
| Bump | bool | Whether a speed bump was nearby |
| Crossing | bool | Whether a road crossing was nearby |
| Give_Way | bool | Whether a give way sign was nearby |
| Junction | bool | Whether a junction was nearby |
| No_Exit | bool | Whether a no exit sign was nearby |
| Railway | bool | Whether a railway was nearby |
| Roundabout | bool | Whether a roundabout was nearby |
| Station | bool | Whether a station was nearby |
| Stop | bool | Whether a stop sign was nearby |
| Traffic_Calming | bool | Whether a traffic calming feature was nearby |
| Traffic_Signal | bool | Whether a traffic signal was nearby |
| Turning_Loop | bool | Whether a turning loop was nearby |
| Sunrise_Sunset | object | Whether the accident happened during day or night |
| Civil_Twilight | object | Light condition based on civil twilight |
| Nautical_Twilight | object | Light condition based on nautical twilight |
| Astronomical_Twilight | object | Light condition based on astronomical twilight |

## 2. Data Cleaning
### 2.1 Missing Values

The dataset was first checked for missing values in each column. Both the number and percentage of missing observations were calculated to determine which variables could be retained and which had too much missing information to be useful for modelling. Decisions on removing or retaining variables were based on the extent of missingness and the usefulness of the affected feature.



In [ ]:

# Record the dataset shape before cleaning
shape_before_cleaning = df.shape

print("Shape before cleaning:", shape_before_cleaning)

# Calculate missing values and percentages
missing_summary = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percentage": (df.isna().mean() * 100).round(2)
})

# Display only columns containing missing values,
# ordered from highest to lowest missing percentage
missing_summary = missing_summary[
    missing_summary["Missing Count"] > 0
].sort_values("Missing Percentage", ascending=False)

display(missing_summary)

Shape before cleaning: (500000, 42)


,Missing Count,Missing Percentage


### 2.2 Duplicate Records

Duplicate rows were checked before further cleaning. Exact duplicate records do not provide additional information to the model and may give repeated observations undue influence during later analysis and modelling. Therefore, any exact duplicate rows identified will be removed while retaining the first occurrence.

In [ ]:
# Check for exact duplicate rows
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 0


In [ ]:
display(missing_summary)

,Missing Count,Missing Percentage
End_Lat,500000,100.00
End_Lng,500000,100.00
Precipitation(in),448054,89.61
Wind_Chill(F),437641,87.53
Wind_Speed(mph),89040,17.81
Visibility(mi),10539,2.11
Weather_Condition,10185,2.04
Humidity(%),7915,1.58
Temperature(F),7250,1.45
Pressure(in),5739,1.15


In [ ]:
# Record the shape before cleaning
shape_before_cleaning = df.shape

# 1. Remove columns with more than 50% missing values (using errors='ignore' to prevent KeyError on rerun)
columns_to_drop = [
    "End_Lat",
    "End_Lng",
    "Wind_Chill(F)",
    "Precipitation(in)"
]

df = df.drop(columns=columns_to_drop, errors='ignore')

# 2. Fill missing values in numerical columns with the median
numeric_columns = df.select_dtypes(include="number").columns

for column in numeric_columns:
    if df[column].isna().any():
        df[column] = df[column].fillna(df[column].median())

# 3. Fill missing values in categorical/text columns with "Unknown"
categorical_columns = df.select_dtypes(
    include=["object", "category", "string"]
).columns

for column in categorical_columns:
    if column == "Weather_Timestamp":
        continue
    if df[column].isna().any():
        df[column] = df[column].fillna("Unknown")

# 4. Fill missing Weather_Timestamp values with its most frequent value
if "Weather_Timestamp" in df.columns and df["Weather_Timestamp"].isna().any():
    df["Weather_Timestamp"] = df["Weather_Timestamp"].fillna(
        df["Weather_Timestamp"].mode()[0]
    )

# 5. Remove exact duplicate rows
df = df.drop_duplicates()

# 6. Record the shape after cleaning
shape_after_cleaning = df.shape

print("Shape before cleaning:", shape_before_cleaning)
print("Shape after cleaning:", shape_after_cleaning)
print("Duplicate rows removed:", shape_before_cleaning[0] - shape_after_cleaning[0])

# 7. Verify remaining missing values
remaining_missing = df.isna().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

print("\nRemaining missing values:")
display(remaining_missing)

Shape before cleaning: (500000, 42)
Shape after cleaning: (500000, 42)
Duplicate rows removed: 0

Remaining missing values:


,0


### 2.3 Missing Values and Duplicate Record

The dataset was examined for missing values in each of the columns, and the number and percentage of missing observations were calculated. A 50% cutoff was used so as to remove columns that provide limited reliable information. Based on this threshold, End_Lat, End_Lng, Wind_Chill(F), and Precipitation(in) were removed because they had 100%, 100%, 87.53%, and 89.61% missing values respectively.
Median values were used so as to fill the relatively smaller amount of missing numerical data. Weather Timestamp was treated separately and its missing values were filled using its most frequent value. This allowed the missing observations to be handled without removing the corresponding accident records. After cleaning, the dataset contained no remaining missing values.
The duplicate check returned 0, meaning that no duplicate rows were found in the original 500,000-row sample which made no observation to be removed because of duplication. The dataset had 500,000 rows and 46 columns before cleaning, and 500,000 rows and 42 columns after cleaning. The row count remained the same because there were no duplicate records and missing values were handled through column removal and imputation.

## 3. Exploratory Data Analysis (EDA)

**3.1** Add this code

In [ ]:
# Check current data types
print("Data types before conversion:")
print(df.dtypes)

Then convert the time-related columns:

In [ ]:
# Convert time columns to datetime
df['Start_Time'] = pd.to_datetime(df['Start_Time'], errors='coerce')
df['End_Time'] = pd.to_datetime(df['End_Time'], errors='coerce')
df['Weather_Timestamp'] = pd.to_datetime(
    df['Weather_Timestamp'], errors='coerce'
)

# Check the new data types
print("\nData types after conversion:")
print(df[['Start_Time', 'End_Time', 'Weather_Timestamp']].dtypes)

Why these conversions are necessary

Start_Time, End_Time, and Weather_Timestamp were originally stored as object rather than datetime. Your dataset documentation identifies these three columns as time-related variables.

Converting them to datetime allows us to extract useful information such as hour, day of week, month, and accident duration.

3.2 Extract Useful Time Features

Use the accident's Start Time to create new time-based features

In [ ]:
# Extract useful time features from Start_Time
df['Start_Hour'] = df['Start_Time'].dt.hour
df['Day_of_Week'] = df['Start_Time'].dt.dayofweek
df['Month'] = df['Start_Time'].dt.month
df['Year'] = df['Start_Time'].dt.year

**You can also create a more readable day name:**

In [ ]:
df['Day_Name'] = df['Start_Time'].dt.day_name()

And calculate how long the accident lasted:

In [ ]:
# Calculate accident duration in minutes
df['Accident_Duration_Minutes'] = (
    df['End_Time'] - df['Start_Time']
).dt.total_seconds() / 60

Then inspect the new features:

In [ ]:
print(df[
    [
        'Start_Time',
        'End_Time',
        'Start_Hour',
        'Day_of_Week',
        'Day_Name',
        'Month',
        'Year',
        'Accident_Duration_Minutes'
    ]
].head())

NameError: name 'df' is not defined

## 4. Preprocessing

## 5. Modelling

## 6. Evaluation